# 00 Run Inventory And Manifest

This notebook inventories every `exp_07_tms_5_2_*` analysis payload currently available, saves a manifest CSV,
and writes publication-style coverage figures so we can see which depth / architecture / checkpoint combinations
are ready for downstream shrinkage analysis.


In [ ]:
from pathlib import Path
import sys

if (Path.cwd() / 'koko_notebooks').exists():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = Path.cwd().resolve().parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from koko_notebooks.shrinkage_analysis.shrinkage_analysis_common import (
    OUTPUTS_DIR,
    PLOTS_DIR,
    RESULTS_DIR,
    batched_expected_masked_weight,
    batched_expected_train_mask_weight,
    collect_ci_outputs,
    component_strengths,
    discover_exp07_analysis_jsons,
    ensure_outputs_dir,
    exhaustive_binary_probe_batch,
    latest_result_per_run,
    layer_weight_metric_row,
    load_component_model_for_checkpoint,
    save_dataframe,
    save_json,
    sampled_probe_batch,
    select_consistent_replicate,
    singleton_probe_batch,
)
from koko_notebooks.shrinkage_analysis.publication_plots import (
    ARCH_COLORS,
    LAYER_COLORS,
    architecture_comparison_plot,
    heatmap,
    histogram_triptych,
    line_plot_by_group,
    line_plot_by_layer,
    multi_metric_panel_by_group,
    multi_metric_panel_by_layer,
    parse_vector_column,
    save_figure,
    setup_publication_style,
    singular_value_trajectory_plot,
)

ensure_outputs_dir()
setup_publication_style()
OUTPUTS_DIR


In [ ]:
import numpy as np
import pandas as pd

CONSISTENT_REPLICATE = 1

manifest_df = latest_result_per_run(discover_exp07_analysis_jsons())
manifest_df = select_consistent_replicate(manifest_df, CONSISTENT_REPLICATE)
manifest_df = manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name']).reset_index(drop=True)
manifest_df[['run_name', 'depth', 'architecture', 'replicate', 'checkpoint_steps']]


In [ ]:
checkpoint_inventory_df = manifest_df[['run_name', 'depth', 'architecture', 'replicate', 'checkpoint_steps']].explode('checkpoint_steps')
checkpoint_inventory_df = checkpoint_inventory_df.rename(columns={'checkpoint_steps': 'checkpoint_step'})
checkpoint_inventory_df['checkpoint_step'] = checkpoint_inventory_df['checkpoint_step'].astype(int)

manifest_csv = save_dataframe(manifest_df, 'csv/run_manifest.csv')
checkpoint_csv = save_dataframe(checkpoint_inventory_df, 'csv/checkpoint_inventory.csv')
manifest_csv, checkpoint_csv


In [ ]:
coverage_counts = checkpoint_inventory_df.groupby(['architecture', 'depth'])['checkpoint_step'].nunique().unstack(fill_value=0)
coverage_counts = coverage_counts.reindex(index=['tied', 'untied']).fillna(0).astype(int)
coverage_counts = coverage_counts.reindex(sorted(coverage_counts.columns), axis=1)

plot_manifest = {}
plot_manifest['coverage_counts'] = heatmap(
    matrix=coverage_counts.to_numpy(),
    row_labels=[str(idx) for idx in coverage_counts.index],
    col_labels=[str(col) for col in coverage_counts.columns],
    title='Available checkpoints per architecture and depth',
    colorbar_label='Checkpoint count',
    subdir='inventory',
    stem='coverage_counts',
    cmap='Blues',
    annotate=True,
    fmt='.0f',
)
plot_manifest


In [ ]:
expected_steps = [5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000]
completeness_rows = []
run_labels = []
for _, row in manifest_df.iterrows():
    run_labels.append(row['run_name'])
    present_steps = set(row['checkpoint_steps'])
    completeness_rows.append([1.0 if step in present_steps else 0.0 for step in expected_steps])
completeness_matrix = np.asarray(completeness_rows, dtype=float)

plot_manifest['checkpoint_completeness'] = heatmap(
    matrix=completeness_matrix,
    row_labels=run_labels,
    col_labels=[f'{step // 1000}k' for step in expected_steps],
    title='Checkpoint completeness by run',
    colorbar_label='Present',
    subdir='inventory',
    stem='checkpoint_completeness',
    cmap='Greens',
    vmin=0.0,
    vmax=1.0,
    annotate=True,
    fmt='.0f',
)

save_json(plot_manifest, 'plots/inventory/manifest.json')
plot_manifest
